<a href="https://colab.research.google.com/github/Apoorvmittal11/23-CS-072-DL-LAB-EXPERIMENT/blob/main/DL%20EXPERIMENT%208/Bidirectional_LSTM_for_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install tensorflow

#**SETUP & DATA LOADING**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
print("Loading IMDB Dataset...")
VOCAB_SIZE = 10000
MAX_LEN = 200
# Load pre-tokenized data
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# Pad sequences
X_train = pad_sequences(X_train, maxlen=MAX_LEN, padding='pre', truncating='post')
X_test = pad_sequences(X_test, maxlen=MAX_LEN, padding='pre', truncating='post')

# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.long)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

# Create DataLoaders
BATCH_SIZE = 64
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Data loaded. Training on: {device}")

Loading IMDB Dataset...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Data loaded. Training on: cpu


#**DEFINING THE BiLSTM MODEL**

In [ ]:
class BiLSTM_Sentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(BiLSTM_Sentiment, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(input_size=embed_dim,
                            hidden_size=hidden_dim,
                            batch_first=True,
                            bidirectional=True)

        # Multiply hidden_dim by 2 because we concatenate forward and backward states
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        out, (hidden, cell) = self.lstm(embedded)

        hidden_fwd = hidden[-2, :, :]
        hidden_bwd = hidden[-1, :, :]

        # Concatenate them along the feature dimension (dim=1)
        final_hidden = torch.cat((hidden_fwd, hidden_bwd), dim=1)
        # final_hidden shape: (batch_size, hidden_dim * 2)

        output = self.fc(final_hidden)

        return output.squeeze(1)

#**INITIALIZATION & TRAINING LOOP**

In [ ]:
EMBED_DIM = 64
HIDDEN_DIM = 64
EPOCHS = 5

model = BiLSTM_Sentiment(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("\n--- Starting Training ---")
for epoch in range(EPOCHS):
    model.train()
    total_loss, total_correct = 0, 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        predictions = model(batch_X)

        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        rounded_preds = torch.round(torch.sigmoid(predictions))
        total_correct += (rounded_preds == batch_y).sum().item()

    avg_loss = total_loss / len(train_loader)
    avg_acc = total_correct / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_loss:.4f} | Train Acc: {avg_acc*100:.2f}%")


--- Starting Training ---
Epoch 1/5 | Train Loss: 0.6452 | Train Acc: 62.09%
Epoch 2/5 | Train Loss: 0.5339 | Train Acc: 74.30%
Epoch 3/5 | Train Loss: 0.4542 | Train Acc: 80.07%
Epoch 4/5 | Train Loss: 0.4384 | Train Acc: 80.43%
Epoch 5/5 | Train Loss: 0.5093 | Train Acc: 74.47%


#**EVALUATION**

In [ ]:
model.eval()
test_loss, test_correct = 0, 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        predictions = model(batch_X)

        loss = criterion(predictions, batch_y)
        test_loss += loss.item()

        rounded_preds = torch.round(torch.sigmoid(predictions))
        test_correct += (rounded_preds == batch_y).sum().item()

final_test_loss = test_loss / len(test_loader)
final_test_acc = test_correct / len(test_loader.dataset)

print("\n--- Final Results ---")
print(f"Test Loss: {final_test_loss:.4f} | Test Accuracy: {final_test_acc*100:.2f}%")


--- Final Results ---
Test Loss: 0.6191 | Test Accuracy: 64.04%
